<a href="https://colab.research.google.com/github/AvanthaPeiris/n8n/blob/master/LanGraph_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==1.5.0 -q
!pip install langchain.openai -q
!pip install langgraph -q
##!pip install -U langchain-google-genai langchain-core -q

In [2]:
!pip install -U langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.9 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [3]:
from typing import TypedDict, List, Union # stacking both user and AI messages via Union
from langchain_core.messages import HumanMessage, AIMessage  # geting both user and AI messages via AIMessage

In [4]:
class agentState(TypedDict):
  msg: List[Union[HumanMessage,AIMessage]]

In [6]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('ChatGPT')

In [8]:
import os
import getpass
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_new')

# #getpass.getpass("Enter your Gemini API key: ")


In [9]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [11]:
def chatBot(state:agentState) -> agentState:
  # Using with_retry on the runnable to automatically retry on transient API errors
  runnable_with_retry = llm.with_retry(
      stop_after_attempt=3,
      wait_exponential_jitter=True
  )
  response = runnable_with_retry.invoke(state['msg'])
  #response = model.invoke(state['msg'])
  state['msg'].append(AIMessage(content=response.content)) # adding AI messages
  print(f'\nAI: {response.content}')
  return state

In [12]:
from langgraph.graph import StateGraph,START,END

workflow=StateGraph(agentState)

workflow.add_node("chatbot",chatBot)

workflow.add_edge(START,"chatbot")
workflow.add_edge("chatbot",END)

app=workflow.compile()




In [13]:
user_prompt=input('Enter your query :')
results = app.invoke({"msg":[HumanMessage(content=user_prompt)]})
print(results)

Enter your query :hi


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4109: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'Hi there! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAVjVB3uilrWRt1SBFoz2XYwUXRznfhMeJ60SAAqwZAnsE/IF2fTsDD1oCSaKzwZhKonupj/7iwbDpuFcmko/oF9RmSwbZMCTvSxk4yNQNuuxgRIjLyAUzYwcHG1zhoTG0elIHA0iy6Oe8EtVzWJFYw+6jlXwvQioIYLulUTu2452t4+HQ+OqPuOS3XA8kjb9X2VS'}}]
{'msg': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': 'Hi there! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAVjVB3uilrWRt1SBFoz2XYwUXRznfhMeJ60SAAqwZAnsE/IF2fTsDD1oCSaKzwZhKonupj/7iwbDpuFcmko/oF9RmSwbZMCTvSxk4yNQNuuxgRIjLyAUzYwcHG1zhoTG0elIHA0iy6Oe8EtVzWJFYw+6jlXwvQioIYLulUTu2452t4+HQ+OqPuOS3XA8kjb9X2VS'}}], additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


In [14]:
chatHistory=[]

user_prompt=input('Enter your query :')
while user_prompt!='exit':
  chatHistory.append(HumanMessage(content=user_prompt))
  results = app.invoke({"msg":chatHistory})
  chatHistory=results['msg']
  #print(results)
  user_prompt=input('Enter your query :')
print("Bye for now! see you soon!")

Enter your query :hi

AI: [{'type': 'text', 'text': 'Hi there! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAVjVB3urUd578+iytRJ4o9D1AOWdJOrPcEWPSyrlkgpx5aK81ZXivtise2Q5BXh4gq3TY5hO8jGrUmyKamD2jqrH6qzszCJfxtK10MewvXK7EZnnIH/5vmlADfuhIlu7p0ExHJiluY8ArcE7Gyh2WICwjYNWeM1Rklr9RE2EuJxDfX5jAG2wUOlODq0YeZv9upTR'}}]
Enter your query :i'm avantha

AI: [{'type': 'text', 'text': 'Nice to meet you, Avantha! How are you doing today?', 'extras': {'signature': 'EpYBCpMBAVjVB3sve94ArnRPRjLeESxXSOaNe41r/UBNtw7BtiXSU6TKsEy0znZm2sxKg27C1qQFcX39GCfBL1+p2JnzJuaRd/lnqgjbZsOmFqXVxQZbGwBFHpxEIWN7eZjF7+3vWx6Ju2VYo9xJzqoayxt7FMgnfWMWhfuYBhez2OQwGbWxT6tnWTfupfXr1oFaRCM2dLue'}}]
Enter your query :who i'm i?

AI: [{'type': 'text', 'text': "You just told me you are Avantha! \n\nAs for who you are beyond that—only you truly know. But based on our chat so far, you're a curious person who just introduced yourself to an AI. How would you describe yourself?", 'extras': {'signature': 'EpYBCpMBAVjVB3vAGpmfN